# PASTIS-R: EDA

PASTIS-R (Panoptic Agricultural Satellite TIme Series) is a **semantic segmentation** benchmark for crop-type mapping. Each sample is a 128×128 Sentinel-2 and Sentinel-1 patch over agricultural land in France, with monthly composites and per-pixel labels across 20 classes (0 = background, 1–18 = crop types, 19 = void).

The published protocol uses 5 spatially-disjoint folds: folds 1–3 train, fold 4 validates, and fold 5 tests. The task is **per-pixel crop-type classification** (19 evaluated classes, background included, void excluded).

## Setup

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# find the repo root by walking up to the folder that has data/
REPO = Path.cwd()
while not (REPO / 'data').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
print('repo:', REPO)

PASTIS = REPO / 'data' / 'input' / 'benchmarks' / 'pastis'
print('PASTIS root:', PASTIS)

# Load metadata
geo = json.loads((PASTIS / 'metadata.geojson').read_text())
rows = [f['properties'] for f in geo['features']]
print(f'Total patches: {len(rows)}')

## Raw layout of one patch

Each patch has three `.npy` files:

| File | Shape | Dtype | Content |
|------|-------|-------|---------|
| `DATA_S2/S2_<id>.npy` | `(T, 10, 128, 128)` | int16 | Sentinel-2 L2A monthly reflectance (10 bands) |
| `DATA_S1A/S1A_<id>.npy` | `(T, 3, 128, 128)` | int16 | Sentinel-1 ascending (VV, VH, VV/VH ratio) |
| `ANNOTATIONS/TARGET_<id>.npy` | `(3, 128, 128)` | uint8 | Semantic (ch 0), boundary (ch 1), instance (ch 2) |

S2 band order (10 bands):

| idx | Band | Description |
|:---:|------|-------------|
| 0 | B2 | Blue (10m) |
| 1 | B3 | Green (10m) |
| 2 | B4 | Red (10m) |
| 3 | B5 | Red Edge 1 (20m) |
| 4 | B6 | Red Edge 2 (20m) |
| 5 | B7 | Red Edge 3 (20m) |
| 6 | B8 | NIR (10m) |
| 7 | B8A | Narrow NIR (20m) |
| 8 | B11 | SWIR 1 (20m) |
| 9 | B12 | SWIR 2 (20m) |

S1 band order (3 channels): index 0 = VV, 1 = VH, 2 = VV/VH ratio.

The loader then monthly-aggregates (mean) by calendar month, producing `(12, C, H, W)` tensors, and slices into 64×64 tiles for the dense encoder pipeline.

In [ ]:
# Inspect one patch
r = rows[0]
pid = int(r['ID_PATCH'])
fold = int(r['Fold'])
print(f'Patch {pid}, Fold {fold}')

s2 = np.load(str(PASTIS / 'DATA_S2' / f'S2_{pid}.npy'))
s1 = np.load(str(PASTIS / 'DATA_S1A' / f'S1A_{pid}.npy'))
target = np.load(str(PASTIS / 'ANNOTATIONS' / f'TARGET_{pid}.npy'))

print(f'S2: {s2.shape} {s2.dtype}  [{s2.min()}..{s2.max()}]')
print(f'S1: {s1.shape} {s1.dtype}  [{s1.min()}..{s1.max()}]')
print(f'TARGET: {target.shape} {target.dtype}  unique classes: {np.unique(target[0])}')

# Dates info
dates = r['dates-S2']
if isinstance(dates, str):
    dates = json.loads(dates)
obs_times = len(dates)
print(f'\nS2 observations: {obs_times}')
print(f'Monthly aggregation: {12} composite timesteps')

## Visualise one patch: RGB composite and segmentation mask

Take the peak-summer month (July / month index 6) for an RGB composite (B4=Red, B3=Green, B2=Blue). Overlay the crop-type mask.

In [ ]:
def _monthly_agg(values, dates_field):
    """Aggregate variable-length observations into 12 calendar months."""
    d = dates_field if isinstance(dates_field, dict) else json.loads(dates_field)
    items = sorted(d.items(), key=lambda kv: int(kv[0]))
    months_idx = np.array([((int(v) // 100) % 100) - 1 for _, v in items])
    out = np.zeros((12,) + values.shape[1:], dtype=np.float32)
    for m in range(12):
        sel = months_idx == m
        if sel.any():
            out[m] = values[sel].mean(axis=0)
    return out

s2_m = _monthly_agg(s2, r['dates-S2'])
# RGB from monthly composite: July = index 6
july = 6
rgb = np.stack([s2_m[july, 2], s2_m[july, 1], s2_m[july, 0]], axis=-1)  # B4, B3, B2
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
ax[0].imshow(rgb)
ax[0].set_title(f'Patch {pid} | Fold {fold} | S2 RGB July (month {july})')

mask = target[0]  # semantic channel
im = ax[1].imshow(mask, cmap='tab20', vmin=0, vmax=19)
ax[1].set_title('Semantic segmentation mask')
plt.colorbar(im, ax=ax[1], shrink=0.8, label='class id')
plt.tight_layout(); plt.show()

## Fold distribution and class balance

The official split is folds 1–3 = train, fold 4 = val, fold 5 = test. Each fold is a spatially-disjoint set of patches.

In [ ]:
import collections

folds = [int(r['Fold']) for r in rows]
fold_counts = collections.Counter(folds)
print('Patches per fold:')
for f in sorted(fold_counts):
    role = {1: 'train', 2: 'train', 3: 'train', 4: 'val', 5: 'test'}.get(f, '?')
    print(f'  Fold {f} ({role}): {fold_counts[f]}')

In [ ]:
# Sample pixel-level class distribution from one patch
labels_flat = target[0].flatten()
class_counts = collections.Counter(labels_flat.tolist())
print(f'Patch {pid} pixel-level class distribution (20 classes, void={class_counts.get(19, 0)} pixels):')
for cid in sorted(class_counts):
    name = {0: 'background', 19: 'void'}.get(cid, f'crop {cid}')
    print(f'  {cid} ({name}): {class_counts[cid]}')

## How the loader transforms a patch: monthly aggregation + tile slicing

The loader monthly-aggregates the raw `(T, C, 128, 128)` observations down to `(12, C, 128, 128)` via mean pooling, then slices into 64×64 tiles (4 tiles per patch), and expands the S2 bands to include NDVI as an 11th channel.

In [ ]:
from evals.benchmarks.pastis_r import load_benchmark as load_pastis, PastisBenchmark

bench = load_pastis(root=REPO / 'data' / 'input' / 'benchmarks', shuffle=False, folds=[5])
print(f'Type: {type(bench).__name__}')
print(f'Patches (fold 5 only): {len(bench.patches)}')
print(f'Virtual tile count: {bench.n_samples}')
print(f'Tile size: {bench.tile_size}x{bench.tile_size}')
print(f'Ignore index (void): {bench.ignore_index}')

# Materialize the first tile
tiles = list(bench.iter_tiles())
tile_id, tile_fold, tile, labels = tiles[0]
print(f'\nFirst tile: {tile_id}')
print(f'  S2: {tile.s2.shape}  S1: {tile.s1.shape}')
print(f'  S2 mask: {tile.s2_mask.shape}  S1 mask: {tile.s1_mask.shape}')
print(f'  Labels shape: {tile.labels.shape}')
print(f'  Valid pixels: {tile.valid.sum()} / {tile.valid.size}')
print(f'  Fold: {tile.fold}')

## Takeaways

One sample is a 12-step S2 (11 bands including NDVI) + S1 (3 bands) patch time series at 64×64 spatial resolution. Label = per-pixel crop type (20 classes). Spatial folds (1–5) define the holdout: train on folds 1–3, validate on fold 4, test on fold 5. Both S2 and S1 modalities are available.